# NQ promotion/suppression profiling

Runs the attack profiler (cell 19 of `robustrag_reproduction.ipynb`) on **Natural Questions** instead of RealtimeQA. No LLM calls: it re-counts keyword votes from the cached isolated responses of an existing NQ reproduction run.

- **Reads** the latest complete `table1-current_v2-open_nq-*` run from the source owner's folder.
- **Writes** `attack-profile-open_nq-alpha*` results to your own output folder.
- No GPU needed: the GPU checks were removed. Everything else in the setup cells is copied unchanged.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, importlib, subprocess
from pathlib import Path
sys.dont_write_bytecode = True
PROJ = "/content/drive/MyDrive/colab/adversarial_rag"
SHARED = "/content/drive/MyDrive/colab/adversarial_rag-shared"
os.chdir(PROJ)
sys.path.insert(0, PROJ)

def run_checked(args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)

# Colab's Python 3.13 cannot install the authors' 2024 package versions.
# Keep the notebook kernel unchanged and run the experiment in Python 3.11.
run_checked([sys.executable, "-m", "pip", "install", "-q", "uv==0.13.5"])
UV = [sys.executable, "-m", "uv"]
ENV_DIR = Path("/content/robustrag-py311")
PYTHON = ENV_DIR / "bin/python"
if not PYTHON.exists():
    run_checked([*UV, "venv", "--python", "3.11", "--managed-python", ENV_DIR])
SPACY_MODEL = ("https://github.com/explosion/spacy-models/releases/download/"
               "en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl")
run_checked([*UV, "pip", "install", "--python", PYTHON,
             "torch==2.2.1", "numpy==1.26.4",
             "transformers==4.40.1", "tokenizers==0.19.1",
             "accelerate==0.29.3", "huggingface-hub==0.23.5",
             "sentencepiece==0.2.0", "joblib==1.3.2",
             "nltk==3.8.1", "spacy==3.7.4", "tqdm==4.66.1",
             "setuptools<81", SPACY_MODEL])
RUN_ENV = dict(os.environ, NLTK_DATA="/content/robustrag-nltk-data",
               PYTHONHASHSEED="0")
os.environ["NLTK_DATA"] = RUN_ENV["NLTK_DATA"]
run_checked([PYTHON, "-m", "nltk.downloader", "-d", RUN_ENV["NLTK_DATA"],
             "stopwords"], env=RUN_ENV)
print("Experiment Python:", PYTHON)

## Personal output folder and source run folder

Enter your name when prompted. The notebook is shared, but experiment outputs remain separated as required by `ONBOARDING.md`. You will also be asked for the folder name that holds the NQ reproduction run (read-only).

In [ ]:
from pathlib import Path
import subprocess

OWNER = input("Output folder name (for example, manya or connor): ").strip()
if not OWNER or OWNER.lower() in {"replace-me", "your-name"}:
    raise ValueError("Enter a non-empty personal output folder name.")

DATA_DIR = Path(SHARED) / "data" / "robustrag"
RUNS_DIR = Path(SHARED) / "outputs" / OWNER / "robustrag-reproduction"
SOURCE_OWNER = input("Folder name holding the NQ reproduction run (for example, connor): ").strip()
SOURCE_RUNS_DIR = Path(SHARED) / "outputs" / SOURCE_OWNER / "robustrag-reproduction"
UPSTREAM_DIR = Path(PROJ) / "third_party" / "robustrag_upstream"
RUNNER_DIR = Path("/content/robustrag-runner")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["ROBUSTRAG_RUNNER_DIR"] = str(RUNNER_DIR)
os.environ["ROBUSTRAG_DATA_DIR"] = str(DATA_DIR)
os.environ["ROBUSTRAG_RUNS_DIR"] = str(RUNS_DIR)
os.environ["ROBUSTRAG_DATA_DIR"] = str(DATA_DIR)
os.environ["ROBUSTRAG_RUNS_DIR"] = str(RUNS_DIR)
os.environ["ROBUSTRAG_RUNNER_DIR"] = str(RUNNER_DIR)
os.environ["NLTK_DATA"] = RUN_ENV["NLTK_DATA"]
os.environ["ROBUSTRAG_SOURCE_RUNS_DIR"] = str(SOURCE_RUNS_DIR)

print("Data:", DATA_DIR)
print("Runs:", RUNS_DIR)
print("Source runs:", SOURCE_RUNS_DIR)

## Prepare the vendored authors' code and verified data

The pinned upstream Python source is committed under `third_party/`, so this step makes no GitHub code download. Shared JSON datasets stay outside git and are downloaded only when missing, with SHA-256 verification.

In [ ]:
from reproduction.robustrag_setup import build_runner, ensure_datasets

build_runner(UPSTREAM_DIR, RUNNER_DIR)
ensure_datasets(DATA_DIR)

preflight = """
import json
import spacy, torch, transformers
from nltk.corpus import stopwords
from src.defense import KeywordAgg
from src.models import create_model
assert transformers.__version__ == '4.40.1'
assert 'the' in stopwords.words('english')
assert spacy.load('en_core_web_sm')('Paris')[0].pos_
print(json.dumps({'python': __import__('sys').version,
                  'torch': torch.__version__,
                  'transformers': transformers.__version__}))
"""
preflight_result = subprocess.run(
    [PYTHON, "-c", preflight], cwd=RUNNER_DIR, env=RUN_ENV,
    check=True, capture_output=True, text=True,
)
print("Preflight:", preflight_result.stdout.strip())
subprocess.run(
    [PYTHON, RUNNER_DIR / "main.py", "--help"], cwd=RUNNER_DIR, env=RUN_ENV,
    check=True, stdout=subprocess.DEVNULL,
)
print("Upstream entry point imports successfully.")

## 5. Promotion/suppression feasibility profiler

This is deliberately a **response-level oracle analysis**, not yet a claim that a natural-looking injected passage can force the desired isolated response.

- **Promotion:** choose a non-gold phrase that the defense actually extracted from a benign response but currently filtered out. Add one vote per injected passage and test whether it crosses the threshold.
- **Suppression:** add distinct semantic non-answers such as 'Unknown marker 1'. RobustRAG counts them as answers because its abstention check only recognizes the literal substring 'I don't'. Because top-k is fixed, this raises the threshold only when the injected passage displaces an abstaining benign passage.
- **Vote loss:** separately record cases where injecting passages removes low-ranked correct support even before the answer-count threshold rises.

The profiler reports changes to the retained keyword set. Gold-aligned phrases are identified with a documented string-matching heuristic and must be manually audited. Structural keyword changes are neither sufficient nor necessary for the final LLM answer to become wrong, so live passage construction and end-to-end validation remain a separate stage. In that stage, first use claim-copying or genuinely claim-free passages; treat any passage-level instruction that forces a chosen response as a separate prompt-injection condition.

In [ ]:
%%script /content/robustrag-py311/bin/python
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import re
import string
import sys

import joblib
import spacy
from nltk.corpus import stopwords

RUNNER_DIR = Path(os.environ["ROBUSTRAG_RUNNER_DIR"])
DATA_DIR = Path(os.environ["ROBUSTRAG_DATA_DIR"])
RUNS_DIR = Path(os.environ["ROBUSTRAG_RUNS_DIR"])
SOURCE_RUNS_DIR = Path(os.environ["ROBUSTRAG_SOURCE_RUNS_DIR"])
DATASETS = {"realtimeqa": ("realtimeqa.json", None), "open_nq": ("open_nq.json", None)}
if str(RUNNER_DIR) not in sys.path:
    sys.path.insert(0, str(RUNNER_DIR))
from src.dataset_utils import load_data
from src.models import BaseModel
from src.prompt_template import MISTRAL_TMPL

NLP = spacy.load("en_core_web_sm")
STOPWORDS = set(stopwords.words("english"))
IGNORED_POS = {"VERB", "INTJ", "ADP", "AUX", "CCONJ", "DET", "PART", "PRON", "SCONJ", "PUNCT", "SPACE"}
SEMANTIC_ABSTENTION = re.compile(r"\b(unknown|cannot determine|can't determine|insufficient information|not enough information|no information|unclear|unavailable)\b", re.I)

def exact_abstention(response):
    """Match the authors' literal abstention test exactly."""
    return "I don't" in response

def extract_keyword_set(response):
    """Mirror the upstream short-answer keyword extractor."""
    if exact_abstention(response):
        return set()
    document = NLP(response)
    phrases = [response.strip()]
    current = []
    for token in document:
        if token.pos_ in IGNORED_POS:
            if current:
                phrases.append("".join(item.lemma_ + item.whitespace_ for item in current).strip())
                phrases.extend(item.lemma_ for item in current)
                current = []
        else:
            current.append(token)
    phrases.append("".join(item.lemma_ + item.whitespace_ for item in current).strip())
    phrases.extend(item.lemma_ for item in current)
    return set(phrases)

def aggregate_keyword_sets(keyword_sets, alpha, beta):
    answering = [keywords for keywords in keyword_sets if keywords]
    threshold = min(beta, alpha * len(answering))
    counts = Counter(keyword for keywords in answering for keyword in keywords)
    retained = {
        keyword for keyword, count in counts.items()
        if count >= threshold and keyword not in string.punctuation and keyword not in STOPWORDS
    }
    return {"answering": len(answering), "threshold": threshold, "counts": counts, "retained": retained}

def normalized_text(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()

def phrase_matches_gold(phrase, answer_aliases):
    phrase_normalized = normalized_text(phrase)
    if not phrase_normalized:
        return False
    for alias in answer_aliases:
        alias_normalized = normalized_text(alias)
        if not alias_normalized:
            continue
        if phrase_normalized == alias_normalized:
            return True
        if len(phrase_normalized) >= 3 and phrase_normalized in alias_normalized:
            return True
        if len(alias_normalized) >= 3 and alias_normalized in phrase_normalized:
            return True
    return False

def load_cached_isolated_responses(run_path, dataset):
    run_path = Path(run_path)
    cache_path = run_path / "cache" / f"mistral7b-{dataset}-10.z"
    if not cache_path.exists():
        raise FileNotFoundError(f"Missing response cache: {cache_path}")
    cache = joblib.load(cache_path)
    filename, _ = DATASETS[dataset]
    data_tool = load_data(dataset, 10, data_path=str(DATA_DIR / filename))
    sample_limit = json.loads((run_path / "manifest.json").read_text())["sample_limit"]
    prompt_builder = BaseModel()
    prompt_builder.prompt_template = MISTRAL_TMPL
    records = []
    for query_index, raw_item in enumerate(data_tool.data[:sample_limit]):
        item = data_tool.process_data_item(raw_item)
        prompts = prompt_builder.wrap_prompt(item, as_multi_choice=False, seperate=True)
        missing = [prompt for prompt in prompts if prompt not in cache]
        if missing:
            raise RuntimeError(f"Query {query_index} is missing {len(missing)} isolated responses from the cache.")
        responses = [cache[prompt] for prompt in prompts]
        records.append({"query_index": query_index, "item": item, "responses": responses})
    return data_tool, records

def load_clean_final_answers(run_path):
    result_files = sorted((Path(run_path) / "result").glob("*keyword*.json"))
    if not result_files:
        return {}
    payload = json.loads(result_files[-1].read_text())
    return {entry["query"]: entry["defended"] for entry in payload if "defended" in entry}

def profile_attack_candidates(run_path, dataset, alpha, beta, max_budget=3):
    data_tool, cached_records = load_cached_isolated_responses(run_path, dataset)
    clean_answers = load_clean_final_answers(run_path)
    findings = []
    profiles = []

    for cached in cached_records:
        item = cached["item"]
        responses = cached["responses"]
        answer_aliases = item["answer"]
        keyword_sets = [extract_keyword_set(response) for response in responses]
        clean = aggregate_keyword_sets(keyword_sets, alpha, beta)
        clean_gold = {word for word in clean["retained"] if phrase_matches_gold(word, answer_aliases)}
        final_answer = clean_answers.get(item["question"])
        clean_correct = None if final_answer is None else data_tool.eval_response(final_answer, item)
        gold_counts = {
            phrase: count for phrase, count in clean["counts"].items()
            if phrase_matches_gold(phrase, answer_aliases)
        }
        wrong_counts = {
            phrase: count for phrase, count in clean["counts"].items()
            if phrase and phrase not in string.punctuation and phrase not in STOPWORDS
            and not phrase_matches_gold(phrase, answer_aliases)
        }
        strongest_wrong = max(wrong_counts, key=lambda phrase: (wrong_counts[phrase], len(phrase), phrase), default=None)
        profiles.append({
            "query_index": cached["query_index"], "question": item["question"],
            "answers": answer_aliases, "clean_correct": clean_correct,
            "answering_passages": clean["answering"],
            "abstaining_passages": len(responses) - clean["answering"],
            "correct_keyword_max_count": max(gold_counts.values(), default=0),
            "gold_aligned_phrase_counts": gold_counts,
            "strongest_incorrect_phrase": strongest_wrong,
            "strongest_incorrect_count": 0 if strongest_wrong is None else wrong_counts[strongest_wrong],
            "clean_threshold": clean["threshold"],
        })

        distractors = {}
        for response in responses:
            if not response.strip() or exact_abstention(response) or SEMANTIC_ABSTENTION.search(response):
                continue
            for candidate in extract_keyword_set(response):
                if not candidate or candidate in string.punctuation or candidate in STOPWORDS:
                    continue
                if phrase_matches_gold(candidate, answer_aliases) or candidate in clean["retained"]:
                    continue
                distractors.setdefault(candidate, response.strip())

        for budget in range(1, max_budget + 1):
            survivors = keyword_sets[:-budget]

            for target, source_response in distractors.items():
                promoted = aggregate_keyword_sets(
                    survivors + [extract_keyword_set(target) for _ in range(budget)],
                    alpha,
                    beta,
                )
                if target in promoted["retained"]:
                    findings.append({
                        "attack": "promotion", "query_index": cached["query_index"],
                        "question": item["question"], "answers": answer_aliases,
                        "clean_correct": clean_correct, "budget": budget, "target": target,
                        "source_response": source_response,
                        "clean_target_count": clean["counts"][target],
                        "clean_threshold": clean["threshold"], "attacked_threshold": promoted["threshold"],
                    })

            filler_sets = [extract_keyword_set(f"Unknown marker {index + 1}") for index in range(budget)]
            survivor_state = aggregate_keyword_sets(survivors, alpha, beta)
            suppressed = aggregate_keyword_sets(survivors + filler_sets, alpha, beta)
            survivor_gold = {word for word in survivor_state["retained"] if phrase_matches_gold(word, answer_aliases)}
            attacked_gold = {word for word in suppressed["retained"] if phrase_matches_gold(word, answer_aliases)}
            if clean_gold and not attacked_gold:
                mechanism = "vote_loss" if not survivor_gold else "threshold_suppression"
                findings.append({
                    "attack": mechanism, "query_index": cached["query_index"],
                    "question": item["question"], "answers": answer_aliases,
                    "clean_correct": clean_correct, "budget": budget,
                    "clean_gold_keywords": sorted(clean_gold),
                    "surviving_gold_keywords": sorted(survivor_gold),
                    "clean_answering": clean["answering"], "attacked_answering": suppressed["answering"],
                    "clean_threshold": clean["threshold"], "survivor_threshold": survivor_state["threshold"],
                    "attacked_threshold": suppressed["threshold"],
                })

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output_dir = RUNS_DIR / f"attack-profile-{dataset}-alpha{alpha}-{timestamp}"
    output_dir.mkdir(parents=True, exist_ok=False)
    (output_dir / "findings.json").write_text(json.dumps(findings, indent=2))
    for profile in profiles:
        profile["labels_by_budget"] = {}
        for budget in range(1, max_budget + 1):
            mechanisms = {
                row["attack"] for row in findings
                if row["query_index"] == profile["query_index"] and row["budget"] == budget
            }
            vulnerable = {name for name in mechanisms if name in {"promotion", "threshold_suppression"}}
            if vulnerable == {"promotion", "threshold_suppression"}:
                label = "both"
            elif vulnerable:
                label = next(iter(vulnerable))
            else:
                label = "safe"
            profile["labels_by_budget"][str(budget)] = {"label": label, "mechanisms": sorted(mechanisms)}
    (output_dir / "profiles.json").write_text(json.dumps(profiles, indent=2))
    summary = {"source_run": str(run_path)}
    for attack in ["promotion", "threshold_suppression", "vote_loss"]:
        matching = [row for row in findings if row["attack"] == attack]
        summary[attack] = {
            "candidate_rows": len(matching),
            "unique_queries": len({row["query_index"] for row in matching}),
            "clean_correct_queries": len({row["query_index"] for row in matching if row["clean_correct"] is True}),
        }
    (output_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))
    print("Saved:", output_dir)
    return output_dir, findings

def review_candidates(findings, attack, limit=15):
    seen = set()
    shown = 0
    for row in findings:
        if row["attack"] != attack or row["query_index"] in seen:
            continue
        seen.add(row["query_index"])
        print("\nQUESTION:", row["question"])
        print("GOLD:", row["answers"])
        print("BUDGET:", row["budget"])
        if "target" in row:
            print("PROMOTION TARGET:", row["target"])
        else:
            print("CLEAN GOLD KEYWORDS:", row["clean_gold_keywords"])
            print("THRESHOLDS:", row["clean_threshold"], "->", row["attacked_threshold"])
        shown += 1
        if shown >= limit:
            break

candidate_runs = []
for candidate in SOURCE_RUNS_DIR.glob("table1-current_v2-open_nq-*"):
    manifest_path = candidate / "manifest.json"
    if manifest_path.exists() and json.loads(manifest_path.read_text()).get("status") == "complete":
        candidate_runs.append(candidate)
if not candidate_runs:
    raise RuntimeError("Complete the current_v2 NQ run before profiling attacks.")
current_rqa_run = max(candidate_runs, key=lambda path: path.stat().st_mtime_ns)
for alpha in (0.2, 0.3):
    rqa_attack_dir, rqa_findings = profile_attack_candidates(
        run_path=current_rqa_run, dataset="open_nq", alpha=alpha, beta=3.0, max_budget=3,
    )
    for attack_name in ("promotion", "threshold_suppression", "vote_loss"):
        print("\n=== alpha", alpha, attack_name, "===")
        review_candidates(rqa_findings, attack_name, limit=15)

## Finish

After confirming that all files have finished writing, unmount Drive. Then run `./pull.sh` locally to download the shared outputs. Before committing this notebook, use **Edit → Clear all outputs** so model logs and environment details are not stored in git.

In [ ]:
drive.flush_and_unmount()